# SCI-JEPA Paper A — local run

Runs the ICML Paper A pipeline on a slice of the normalized arXiv parquet, with the paper's fixed configuration:

1. **Build the knowledge-unit (KU) graph** from `data/normalized/arxiv` (cached after the first build)
2. **Look at the data**: KU counts, cleaning outcomes, one paper's KUs
3. **Live dashboard + animated graphs** (HTML): one paper's subgraph, how a training instance is built and masked, and the corpus graph
4. **Shape audit** (Theorem A2, no training)
5. **One training run**, streamed live to the dashboard: losses, per-KU-type learning, collapse monitor, gradients, learned relation weights, validation curves, neuron-by-neuron interpretation, a plain-language Explain tab, then where it learns / fails with real examples
6. **Experiments** (`BENCH` by default: the text baseline and every main model variant, ranked in the dashboard's **Benchmark** tab; every other experiment code is listed in the Parameters cell): every arm × seed streams to the dashboard; the *Compare* tab puts two runs side by side in plain words, the *All runs* tab lists every model live

Dashboard tabs: **World model** (does the model use the subgraph's links, can it predict the effect of actions, and one subgraph seen through the model's eyes) · **Overview** (key numbers + top findings) · **Explain** (plain-language reading of the run, glossary) · **Training dynamics** · **Validation (live)** · **Neurons** (what every neuron encodes, which ones carry the paper-identity shortcut or a leak) · **Where it learns** · **Examples** · **Graphs** · **Reasoning tasks** · **Compare** (two runs side by side: which is better at what) · **All runs**. All curves can be plotted against **epoch** or step; dotted lines mark epoch boundaries and sequential-masking phase ends.

**Before the first run:** in a terminal, from the repository root, `bash scripts/run_local.sh setup`, then pick the kernel **Python (sci-jepa)**. Data: `docs/data.md`.

Change only the **Parameters** cell (13 settings). Research domains are balanced when the corpus is built, and every score is also macro-averaged over domains (the primary reasoning index), so large fields such as Physics or Computer Science cannot dominate. Every experiment writes **paper figures** (PDF + PNG for every chart and every run, plus a LaTeX leaderboard table) to `runs/paper_a_results/<experiment>/figures/`, its **main findings** (pre-registered hypotheses) and a **PDF report**. `N_PAPERS = 500` checks that everything works; 5000 gives local results. `SPEED = "fast"` (default) runs the same model faster: fp16 text encoder and training, length-sorted padding, batches built in the background (8.4x faster embedding, 1.2x-1.8x faster training, identical accuracy in the `SPEED` experiment).

## Parameters

Only what changes between runs. Everything else is the ICML paper's fixed configuration, printed in section 1.

In [ ]:
N_PAPERS     = 500         # papers: 500 = local benchmark run, 5000 = local results; the paper's D0 = 60000
                          #   (MacBook with SPEED="fast": ~1.6 h to embed D0, 500000 papers ~14 h)
SIZE         = "local"     # "local" = small (d 256, 4+2 layers, 5 M) | "S" = the paper's model (d 512, 8+6, 46 M)
                          # | "M" = medium (d 768, 12+8, ~130 M); S and M are practical on a MacBook with SPEED
EPOCHS       = 40          # MAXIMUM passes over every hidden KU. Every epoch is evaluated on validation; a run stops
                          #   when the validation reasoning index has not improved for PATIENCE epochs, and the test
                          #   uses the best epoch's weights. Measured on 500 real papers: best epochs 13-31 (GNN 13,
                          #   paper-as-text 15, world model 18, reference / motif 19, motif + world 31), all
                          #   stopped by epoch 36, so 40 covers every model; a fixed 10 would cut all of them short
PATIENCE     = 5           # early stopping: epochs without improvement before a run stops (0 = run all EPOCHS)
SEEDS        = [0, 1, 2]   # the paper reports 5 seeds: [0, 1, 2, 3, 4]. Time at 500 papers (MacBook, SPEED="fast"):
                          #   BENCH ~2-3 h per seed (+ the LLM arm), ALL ~12-15 h per seed; start with [0], add seeds
                          #   later (finished runs are kept and skipped)
EXPERIMENTS  = ["BENCH"]   # what to run after the reference model; [] = nothing. See the Benchmark tab.
# "BENCH"  benchmark: 6 comparison systems (frozen text, BM25, SciNCL, bge, supervised regressor, LLM-Reasoning),
#          JEPA with 3 GNN encoders (R-GAT, GINE, GraphGPS) + Graph-JEPA, and the 5 JEPA graph-transformer
#          variants (15 arms; the LLM is set below; SciNCL / bge are downloaded once, ~0.5 GB each)
# "ALL"    every distinct method of every experiment below (67 arms; long: use few papers / seeds first)
# Appendix runs (one by one):
# "APPX_STRUCT"  random-walk structural encodings, alone and with motif + world model            4 arms
# "APPX_EVAL"    old vs redesigned evaluation (topic-matched candidates, counterfactuals, masking) 5 arms
# "SPEED"        MacBook speed settings vs the reference: same accuracy? (fast, multi-target)    9 arms
# Single experiments, to run one by one (e.g. EXPERIMENTS = ["E10"]):
# "E10"    more context papers: N = 1, 4, 8, 16 (does it remove the identity shortcut?)      4 arms
# "E11"    inverse-relation head: lambda_inv = 0, 0.1, 0.3, 1                                 4 arms
# "E12"    grouping regulariser: off / within a paper / across papers                         3 arms
# "E4"     graph construction: frozen vs resampled neighbours x node count shown or not       4 arms
# "E5"     contrast masking: off / on                                                         2 arms
# "E15"    one model for all KU types vs one model per type                                  11 arms
# "E17"    paper as separate KUs vs paper as one text                                         2 arms
# "E18"    E15 with the paper read as one text                                               11 arms
# "E19"    JEPA world model: learns how graph actions change a hidden KU                      3 arms
# "E20"    Graph-JEPA ideas: motif targets, structural encodings, + world model               6 arms
# "E1" "E2" "E3"  SHACL shapes / t-norms (placeholder shapes until the ORKG shapes exist)   3 / 9 / 4 arms
# Runs = arms x len(SEEDS); every run streams to the dashboard (Benchmark, Compare, All runs tabs).
SPEED        = "fast"      # "fast" = same model and accuracy, 1.2x (small) to 1.8x (M) faster training and 8x faster
                          #          text embedding (fp16 + background batch building)
                          # "fast+multi" = + multi-target training (4 hidden KUs per instance): 4x faster but
                          #          LOSES accuracy in the SPEED check (reasoning index 3.3 vs 4.25); not recommended
                          # "off" = the original code path
LLM_MODEL    = "Qwen/Qwen2.5-1.5B-Instruct"   # LLM-Reasoning baseline (any Hugging Face causal LM)
                          #   MacBook (<= ~2B): "Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct",
                          #     "meta-llama/Llama-3.2-1B-Instruct" (gated: accept the licence on Hugging Face)
                          #   GPU server: "Qwen/Qwen2.5-7B-Instruct", "meta-llama/Llama-3.1-8B-Instruct",
                          #     "Qwen/Qwen2.5-32B-Instruct", "Qwen/Qwen2.5-72B-Instruct" (multi-GPU)
LLM_ON_MAC   = True        # run LLM-Reasoning without a CUDA GPU (Apple GPU, fp32). False = skip it here
LLM_QUERIES  = 300         # queries per retrieval task for the LLM (each ranks ~100 candidates);
                          #   1.5B on an M5 Pro: ~10 s per query -> ~3-4 h for the LLM arm; GPU server: 600+
FIELD_BALANCE = "temperature"  # research-domain balance of the corpus (arXiv: Physics 36%, CS 25%, Math 18%, ...):
                          #   "temperature" = domain share ~ size^FIELD_ALPHA | "uniform" = equal | "none" = as sampled
FIELD_ALPHA  = 0.5         # 1 = natural shares, 0.5 = square root (small domains get enough test papers), 0 = equal
OPEN_BROWSER = True        # open the live dashboard in the browser

## Setup

In [ ]:
import os, sys, json, time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # the repository root
os.environ.setdefault("PAPER_A_PARQUET", str(REPO / "data/normalized/arxiv"))
os.environ.setdefault("PAPER_A_INDEX_DIR", str(REPO / "runs/paper_a_index"))
os.environ.setdefault("PAPER_A_OUT", str(REPO / "runs/paper_a_results"))
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# always load the CURRENT code: drop sci_jepa modules cached by an earlier run of this kernel
# (otherwise edits to src/sci_jepa/*.py only take effect after a kernel restart)
for _m in [m for m in sys.modules if m == "sci_jepa" or m.startswith("sci_jepa.")]:
    del sys.modules[_m]

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from sci_jepa.config import (PaperAConfig, SIZES, EXPERIMENTS as EXP_ARMS, ICML_FIXED,
                             PAPER_EXPERIMENTS, resolve_device, FAST, FAST_MULTI)
from sci_jepa.data import ku_schema
from sci_jepa.data.graph_index import get_index, HUB
from sci_jepa.data.instances import Sampler
from sci_jepa.models.shapes import SlotLabels, audit_tightness
from sci_jepa.training.train import train_one, run_experiment
from sci_jepa.dashboard.telemetry import LiveRun
from sci_jepa.reporting import graph_viz as V
from sci_jepa.dashboard.serve import setup_dashboard, start_server, register_graph
from IPython.display import IFrame, display, HTML
import webbrowser

assert Path(os.environ["PAPER_A_PARQUET"]).is_dir(), "parquet folder not found: set PAPER_A_PARQUET"
print("torch", torch.__version__, "| device ->", resolve_device("auto"))
print("parquet :", os.environ["PAPER_A_PARQUET"])
print("results :", os.environ["PAPER_A_OUT"])

## 1. The ICML configuration

Every run uses this configuration (the defaults in `train/paper_a/config.py`); each experiment changes exactly one thing.

In [ ]:
bad = [e for e in EXPERIMENTS if e not in PAPER_EXPERIMENTS]
assert not bad, f"not an ICML experiment: {bad}; choose from {list(PAPER_EXPERIMENTS)}"
cfg = PaperAConfig().override(**SIZES[SIZE], max_papers=N_PAPERS, epochs=float(EPOCHS), patience=PATIENCE, seeds=list(SEEDS),
                              device="auto", batch_size=64 if SIZE in ("tiny", "local") else 128,
                              rows_per_shard=100 if N_PAPERS <= 20000 else 500,
                              llm_model=LLM_MODEL, llm_allow_cpu=LLM_ON_MAC, llm_eval_instances=LLM_QUERIES,
                              field_balance=FIELD_BALANCE, field_alpha=FIELD_ALPHA,
                              **{"off": {}, "fast": FAST, "fast+multi": FAST_MULTI}[SPEED])
display(pd.Series({**ICML_FIXED, "papers": f"{N_PAPERS} (model size {SIZE}, {len(SEEDS)} seeds)",
                   "speed": f"{SPEED}: amp={cfg.amp}, prefetch={cfg.prefetch}, targets/instance={cfg.targets_per_instance}, "
                            f"text encoder {cfg.embed_dtype}, padding {cfg.embed_bucket}"}).to_frame("fixed configuration"))
display(pd.DataFrame({"question": PAPER_EXPERIMENTS, "arms": {k: ", ".join(t for t, _ in EXP_ARMS[k]) for k in PAPER_EXPERIMENTS},
                      "run now": {k: "yes" if k in EXPERIMENTS else "" for k in PAPER_EXPERIMENTS}}))

## 2. Build (or load) the KU graph

The first build samples 100 random papers from each of `N_PAPERS / 100` parquet shards and embeds every KU with gte-modernbert (8,192-token context, every KU read in full): with `SPEED = "fast"` about 1.7 minutes per 1,000 papers on the Apple GPU (about 8 minutes for 5,000; with `SPEED = "off"` about 14 and 70 minutes, measured: 7 minutes for 500). The fp16 text encoder gives its own cache (vectors differ from fp32 only in the 5th decimal). The embedding is saved in chunks as it goes: if a long build stops, run the cell again and it continues where it stopped. E17 / E18 also build the paper-as-text graph (about 3 s per paper). The build prints how many texts were truncated (should be 0). Later runs load the cache from `paper_a_data/`.

In [ ]:
t0 = time.time()
gi = get_index(cfg)
S = gi.summary()
print(f"{S['papers']} papers, {S['kus']} KUs ({S['kus_per_paper']} per paper) in {time.time() - t0:.0f}s")
print("splits:", S["splits"], "| fields:", S["fields"], "| subfields:", S["subfields"])
print("in-corpus citations:", S["paper_edges_cites"], "| proxy grounded edges:", S["paper_edges_proxy"])
from sci_jepa.training.train import mask_plan      # run length from the data: EPOCHS passes over every hidden KU
n_steps, _, sched = mask_plan(cfg, Sampler(gi, cfg, "train", seed=0))
cfg = cfg.override(steps=n_steps, log_every=max(n_steps // 50, 1), eval_every=max(n_steps // 10, 1),
                   warmup=min(1000, max(n_steps // 10, 1)))
print(f"at most {EPOCHS} epochs over {sum(sched['anchors_per_type'].values())} hidden KUs = {n_steps} steps of {cfg.batch_size}"
      + (f" (early stopping after {PATIENCE} epochs without improvement)" if PATIENCE else ""))
if n_steps < 1000:
    print("WARNING: fewer than 1,000 training steps -- too few for the predictor to leave the average; "
          "use more papers or epochs before reading the results")
pd.DataFrame({"KUs": S["per_type"], "per paper": {k: round(v / S["papers"], 2) for k, v in S["per_type"].items()}})

Cleaning outcomes per KU field: what became a KU (`ok`) and why the rest did not. `null_statement` = text such as *"The paper does not present any contradicting evidence."*

In [ ]:
out = pd.Series(S["build"]["ku_outcomes"]).rename("count").to_frame()
out.index = pd.MultiIndex.from_tuples([tuple(i.split(":")) for i in out.index], names=["field", "outcome"])
tab = out["count"].unstack(fill_value=0)
tab["ok %"] = (100 * tab.get("ok", 0) / tab.sum(axis=1)).round(1)
tab

Years and fields in the selected papers:

In [ ]:
yrs = pd.Series(gi.paper_year[gi.paper_year > 0]).value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
yrs.plot.bar(ax=ax[0], title="papers per year"); ax[0].axvline(list(yrs.index).index(cfg.time_T) if cfg.time_T in yrs.index else 0, c="r", ls="--")
pd.Series([gi.field_names[f] for f in gi.paper_field]).value_counts().head(15).plot.barh(ax=ax[1], title="top fields")
plt.tight_layout(); plt.show()

### One paper's knowledge units

In [ ]:
p = int(gi.papers("train")[0])
rows = [{"type": ku_schema.COMP_TYPES[gi.comp_type[c]],
         "quantitative": bool(gi.comp_quantitative[c]), "cites_figure": bool(gi.comp_cites_figure[c]),
         "challenged": bool(gi.comp_challenged[c])} for c in gi.paper_comps[p]]
print("paper:", gi.paper_ids[p], "| year", gi.paper_year[p], "| field", gi.field_names[gi.paper_field[p]],
      "| type", ku_schema.PAPER_TYPES[gi.paper_type[p]])
edges = [(ku_schema.COMP_TYPES[gi.comp_type[c]], ku_schema.ROLE_RELS[r - 3], ku_schema.COMP_TYPES[gi.comp_type[d]])
         for c in gi.paper_comps[p] for d, r in gi.comp_edges.get(int(c), []) if 3 <= r < 3 + len(ku_schema.ROLE_RELS)]
print("role edges:", pd.Series(edges).value_counts().to_dict())
pd.DataFrame(rows).value_counts("type").to_frame("KUs")

## 3. Live dashboard and animated graphs

Starts a small local web server (browsers block live updates from `file://` pages) and writes three kinds of animated HTML pages to `paper_a_results/dashboard/graphs/`:

| page | what it shows |
|---|---|
| **paper subgraph** | one paper's KUs appearing one by one, joined by the schema edges (context→question→method→result→claim→evidence/implication) |
| **training instance** (claim / result / method) | step by step: the anchor paper → its neighbourhood of N papers → the target masked → its paraphrases hidden → probe edges hidden. The caption shows the nuisance audit |
| **corpus graph** | papers appearing one by one, coloured by field, linked by citations, proxy grounded edges and shared subfields; plus a KU-level view |

Every page has Play / Pause / Step / Reset, a stage slider, physics on/off, and hover tooltips with the KU text. The dashboard's **Graphs** tab embeds them all.

In [ ]:
DASH = setup_dashboard(os.path.join(os.environ["PAPER_A_OUT"], "dashboard"))
URL = start_server(DASH, 8765)
G = os.path.join(DASH, "graphs")

# one paper (the train paper with the most KUs, so every type shows up)
p_show = max(gi.papers("train")[:500], key=lambda p: len(gi.paper_comps[int(p)]))
V.paper_html(gi, int(p_show), f"{G}/paper_subgraph.html")
register_graph(DASH, f"Paper subgraph ({gi.paper_ids[int(p_show)]})", "graphs/paper_subgraph.html")

# one training instance per target type, built exactly as the trainer builds them
for t in cfg.target_types:
    inst = Sampler(gi, cfg.override(target_types=(t,)), "train", seed=1).sample(1)[0]
    V.instance_html(gi, inst, cfg, f"{G}/instance_{t}.html")
    register_graph(DASH, f"Training instance: masked {t} (N={cfg.n_subgraphs})", f"graphs/instance_{t}.html")

V.corpus_html(gi, f"{G}/corpus_papers.html", n_papers=150, split="train")
register_graph(DASH, "Corpus graph: 150 papers", "graphs/corpus_papers.html")
V.corpus_html(gi, f"{G}/corpus_kus.html", n_papers=30, ku_level=True, split="train")
register_graph(DASH, "Corpus graph: 30 papers with their KUs", "graphs/corpus_kus.html")

print("dashboard :", URL)
print("graphs    :", G)
for g in sorted(os.listdir(G)):
    if g.endswith(".html"):
        print("   ", URL + "graphs/" + g)
if OPEN_BROWSER:
    webbrowser.open(URL)
display(IFrame(URL + "graphs/instance_claim.html", width="100%", height=620))

## 4. Shape audit (Theorem A2, no training)

Tight = the admissible set given the context is small relative to the slot's content entropy, and the context does not fix it. Placeholder vocabulary until the ORKG shapes arrive.

In [ ]:
pd.DataFrame(audit_tightness(cfg.shape_spec, SlotLabels(gi), gi))

## 5. One training run (live)

The reference configuration (plain JEPA objective, every λ = 0), seed `SEEDS[0]`. While this cell runs, the dashboard updates every few seconds: **Training dynamics** (losses, loss per KU type, prediction↔target agreement, collapse monitor, gradient norms, learned relation weights) and **Validation (live)** (retrieval vs chance overall and per KU type, chi, effective rank, paper-identity probe). **Neurons** labels every encoder neuron and predictor output at each validation pass, and **Explain** turns all of it into plain-language verdicts. When it finishes, **Where it learns** and **Examples** fill in from the test split, and the Neurons tab shows a final report on the test split.

In [ ]:
RUN_NAME = f"single__N{cfg.n_subgraphs}__" + time.strftime("%Y%m%d_%H%M%S")
live = LiveRun(DASH, RUN_NAME, cfg, gi, extra={"experiment": "single", "seed": SEEDS[0]})
print(f"streaming to {URL}?run={RUN_NAME}  (open it now and watch the curves move)")
display(HTML(f'<a href="{URL}?run={RUN_NAME}" target="_blank">open the live dashboard for this run ↗</a>'))
t0 = time.time()
MODEL_DIR = os.path.join(os.environ["PAPER_A_OUT"], "models", RUN_NAME)
m = train_one(cfg, gi, SEEDS[0], live=live, save_dir=MODEL_DIR)   # the trained model is saved here
print(f"trained {cfg.steps} steps in {time.time() - t0:.0f}s on {resolve_device(cfg.device)}")

h = pd.DataFrame(m["history"]).set_index("step")
h[[c for c in h.columns if c not in ("sec", "flagged")]].plot(logy=True, figsize=(7, 3), title="training losses"); plt.show()

def flat(m):
    def g(*path):                                  # missing values (e.g. the no-training baseline) -> NaN
        d = m
        for k in path:
            d = d.get(k) if isinstance(d, dict) else None
        return float("nan") if d is None else d
    return {"balanced MRR": g("retrieval_balanced", "MRR"), "balanced chance": g("retrieval_balanced", "chance_MRR"),
            "balanced Hits@10": g("retrieval_balanced", "Hits@10"),
            "standard MRR": g("retrieval_standard", "MRR"), "standard chance": g("chance_MRR_standard"),
            "shortcut margin (MRR)": g("shortcut_margin_MRR", "mean"),
            "chi (target, type-centred)": g("target_space", "chi"), "chi raw": g("target_space", "chi_raw"),
            "eff. rank (target)": g("target_space", "eff_rank"),
            "partition probe acc": g("partition_probe", "probe_acc"),
            "partition probe chance": g("partition_probe", "probe_chance"),
            "between-subgraph var share": g("partition_probe", "between_subgraph_var_share"),
            "relation sensitivity": g("relation_sensitivity"), "inverse-relation acc": g("inverse_acc"),
            "audit: flagged frac": g("audit", "flagged_frac"), "audit: mean max ctx sim": g("audit", "mean_max_ctx_sim"),
            "train sec": g("train_sec"), "trainable params": g("params_trainable")}
pd.Series(flat(m)).to_frame("value")

### Epoch by epoch (single run)

Validation at the end of every epoch: the same numbers as the dashboard's **Epochs** tab. `×chance ≥ 1.5` = learned.

In [ ]:
def epoch_table(m):
    rows = []
    for r in m.get("epoch_results", []):
        rows.append({"epoch": r["epoch"], "train JEPA": r["train_jepa"], "val balanced MRR": r["bal_MRR"],
                     "chance": r["bal_chance"], "x chance": r["bal_MRR"] / max(r["bal_chance"], 1e-9),
                     "Hits@10": r["bal_H10"], "chi": r["chi"], "probe": r["probe_acc"],
                     "types learned": f"{r['types_learned']}/{len(r['per_type'])}",
                     "identity neurons": (r.get("neurons") or {}).get("identity"),
                     "predictor dims learned": (r.get("neurons") or {}).get("pred_learned"), "sec": r["sec"]})
    return pd.DataFrame(rows).set_index("epoch") if rows else pd.DataFrame()

def epoch_types(m):
    R = m.get("epoch_results", [])
    return pd.DataFrame({f"epoch {r['epoch']}": {t: v["MRR"] / max(v["chance"], 1e-9) for t, v in r["per_type"].items()}
                         for r in R}).round(2)

et = epoch_table(m)
if len(et):
    display(et.round(4))
    print("balanced MRR / chance for every KU type at every epoch:")
    display(epoch_types(m).style.background_gradient(cmap="RdYlGn", vmin=0.8, vmax=2.0))
    fig, ax = plt.subplots(1, 2, figsize=(12, 3))
    ax[0].plot(et.index, et["val balanced MRR"], "o-", label="val balanced MRR"); ax[0].plot(et.index, et["chance"], "k:", label="chance")
    ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(et.index, et["train JEPA"], "o-", c="C3"); ax[1].set_yscale("log"); ax[1].set_xlabel("epoch"); ax[1].set_title("train JEPA loss (epoch mean)")
    plt.tight_layout(); plt.show()
else:
    print("no end-of-epoch results: set EPOCHS >= 1")

### Reasoning-task evaluation matrix (single run)

The same model queried five ways (final model, test split). Slots map to KU types: Problem = context + question, Method = method, Material / Condition = procedure, Result = result, Claim / Hypothesis = claim (+ question). Retrieval tasks: balanced MRR ÷ chance (≥ 1.5 = solved). Consistency: balanced accuracy, chance 0.5. Intervention: relative shift of the predicted result when the condition / method is swapped for another paper's (0 = ignores it), and the share of swaps that move it toward the donor's actual result (chance 0.5).

In [ ]:
from sci_jepa.evaluation.tasks import TASK_ORDER, TASK_LABEL

def task_table(tasks):
    if not tasks or "note" in tasks:
        return pd.DataFrame({"note": [tasks.get("note", "no task results") if tasks else "no task results"]})
    rows = {}
    for k in ("forward", "inverse", "abduction"):
        t = tasks[k]
        rows[TASK_LABEL[k]] = {"query": t["query"], "score": t.get("ratio"), "metric": "MRR / chance",
                               "MRR": t.get("MRR"), "chance": t.get("chance"), "Hits@10": t.get("Hits@10"), "n": t["n"],
                               "by target": ", ".join(f"{g} {b['ratio']:.2f}x" for g, b in t.get("by_target", {}).items())}
    c = tasks["consistency"]
    rows[TASK_LABEL["consistency"]] = {"query": c["query"], "score": c.get("bal_acc"), "metric": "balanced acc.",
                                       "chance": 0.5, "n": c["n"],
                                       "by target": f"supports recall {c.get('recall_supports')}, contradicts recall {c.get('recall_contradicts')}"}
    iv = tasks["intervention"]
    for name in ("condition", "method"):
        v = iv[name]
        rows[f"{TASK_LABEL['intervention']} ({name})"] = {"query": iv["query"], "score": v.get("rel_shift"),
                                                          "metric": "relative shift", "n": v["n"],
                                                          "by target": f"toward donor {v.get('toward_donor')}, away from own {v.get('away_from_own')}"}
    return pd.DataFrame(rows).T

display(task_table(m.get("tasks")))
R = [r for r in m.get("epoch_results", []) if r.get("tasks") and "note" not in r["tasks"]]
if R:
    from sci_jepa.evaluation.tasks import task_summary
    print("every task at every epoch (validation):")
    display(pd.DataFrame({f"epoch {r['epoch']}": task_summary(r["tasks"]) for r in R}).round(3))

**Reading the numbers**
- *balanced MRR* vs *balanced chance*: can the model pick the true masked KU from 100 same-type KUs of related papers? This is the primary metric.
- *chi*: within-subgraph variance share (H-JEPA scale: collapsed ≈ 0.01, healthy ≈ 0.12). *partition probe*: how decodable paper identity is (G1 wants the between-subgraph variance share below 0.30).
- *audit flagged*: fraction of instances in which a visible KU is ≥ 0.75 cosine to the hidden target, i.e. a paraphrase leak the masking could not remove.

## 6. Experiments (live)

Each arm × seed streams to the dashboard as its own run (with *follow latest* on, the dashboard switches to the run that is training now); the **All runs** tab lists every model with its epoch, loss, validation MRR vs chance, identity neurons and a verdict, and overlays any metric across runs. Each experiment trains every arm × every seed (`len(arms) × len(SEEDS)` runs) and writes `paper_a_results/<exp>/`. With ≥ 2 seeds, `summary.json` adds paired t-tests against the first arm.

In [ ]:
for e in EXPERIMENTS:
    print(e, "->", [tag for tag, _ in EXP_ARMS[e]])
print("runs:", sum(len(EXP_ARMS[e]) for e in EXPERIMENTS) * len(SEEDS), f"x {EPOCHS} epochs (one-type runs are shorter: fewer hidden KUs)")

In [ ]:
all_results = {}
for e in EXPERIMENTS:
    print("#" * 30, e, "#" * 30)
    res, summ = run_experiment(cfg, gi, e, log=print, live_root=DASH)
    from sci_jepa.reporting.report_pdf import build as build_pdf      # leaderboards + method catalogue as PDF
    print("PDF report:", build_pdf(os.path.join(os.environ["PAPER_A_OUT"], e)))
    print("paper figures (PDF + PNG) and leaderboard.tex:", os.path.join(os.environ["PAPER_A_OUT"], e, "figures"))
    all_results[e] = res
    rows = []
    for tag, runs in res.items():
        for r in runs:
            rows.append({"exp": e, "arm": tag, "seed": r["seed"], **flat(r)})
    display(pd.DataFrame(rows).groupby(["exp", "arm"], sort=False).mean(numeric_only=True)
            [["balanced MRR", "balanced chance", "standard MRR", "chi (target, type-centred)",
              "partition probe acc", "inverse-relation acc", "audit: flagged frac", "train sec"]].round(4))

### 6a. Benchmark validity, research domains and main findings

- **Surface-cue audit** (`validity.audit`): a classifier that sees only the true text and its counterfactual edit, without the paper, tries to name the original. Edits it detects from wording alone are removed (`mp_filter`, on by default); a whole kind is removed when it is detected in ≥ 90% of cases (on 500 real papers: negation, 100%).
- **Faithfulness** (`validity.faithfulness`): are the LLM-generated knowledge units supported by the paper's full text?
- **Expert annotation** (`validity.export` / `validity.agreement`): sheets for two annotators and their agreement (Cohen's kappa).
- **Research domains**: the corpus is domain-balanced when it is built, and the primary reasoning index is macro-averaged over domains (each domain weighs the same).
- **Contamination**: every task is also scored for test papers after the text encoders' pretraining cutoff (`by_period`).
- **Main findings** (`findings.py`): pre-registered hypotheses H1–H5 with bootstrap confidence intervals and a written conclusion; also in the dashboard, the PDF and the paper figures.


In [ ]:
from sci_jepa.evaluation import validity as VAL
from sci_jepa.evaluation.findings import analyse

print("research domains in this corpus:", gi.build_stats.get("domains"))
if gi.build_stats.get("domain_balance"):
    b = gi.build_stats["domain_balance"]
    pool, sel = b["pool_by_domain"], b["selected_by_domain"]
    if any(isinstance(v, dict) for v in pool.values()):          # stored split: counts per split
        display(pd.DataFrame({f"{s} {k}": d.get(s, {}) for s in pool for k, d in (("candidates", pool), ("selected", sel))}).fillna(0).astype(int))
    else:
        display(pd.DataFrame({"candidates": pool, "selected": sel}).fillna(0).astype(int))
aud = VAL.audit(gi, cfg); aud.pop("flagged")
display(pd.DataFrame(aud["by_kind"]).T.rename_axis("counterfactual kind"))
print("removed kinds:", aud.get("removed_kinds"), "| share of edits removed:", f"{aud['flagged_share']:.0%}")
fa = VAL.faithfulness(gi, cfg, n=60); fa.pop("lowest", None)
if fa.get("n"):
    display(pd.DataFrame(fa["by_type"]).T.round(2).rename_axis("KU type"))
ann = VAL.export(gi, cfg, os.path.join(os.environ["PAPER_A_OUT"], "annotation"), n=200)
print("annotation sheets for two experts:", ann)
for e in EXPERIMENTS:                                   # main findings of every experiment run above
    F = analyse(os.path.join(os.environ["PAPER_A_OUT"], e))
    if F.get("hypotheses"):
        display(pd.DataFrame([{k: h.get(k) for k in ("id", "claim", "a", "b", "difference", "ci95", "outcome")}
                              for h in F["hypotheses"]]).set_index("id"))
        print(f"{e} conclusion:", F["conclusion"])


### Leaderboards (test split)

The same two leaderboards as the dashboard's **Benchmark** tab: mean ± std over seeds, one row per arm. **Reasoning index** = mean of the five reasoning scores on one × chance scale (consistency and intervention ÷ 0.5).

In [ ]:
def _g(d, *path):
    for k in path:
        d = d.get(k) if isinstance(d, dict) else None
    return d

def reasoning_row(m):
    T = m.get("tasks") or {}
    sc = {"forward": _g(T, "forward", "ratio"), "experiment design": _g(T, "inverse", "ratio"),
          "abduction": _g(T, "abduction", "ratio"), "consistency (bal. acc)": _g(T, "consistency", "bal_acc"),
          "intervention: toward new result": _g(T, "intervention", "condition", "toward_donor"),
          "intervention: shift": _g(T, "intervention", "condition", "rel_shift")}
    v = [sc["forward"], sc["experiment design"], sc["abduction"],
         None if sc["consistency (bal. acc)"] is None else sc["consistency (bal. acc)"] / 0.5,
         None if sc["intervention: toward new result"] is None else sc["intervention: toward new result"] / 0.5]
    v = [x for x in v if x is not None and x == x]
    return {"reasoning index": float(np.mean(v)) if v else float("nan"), **sc}

def health_row(m):
    rb = m["retrieval_balanced"]; st = m.get("structure") or {}; ac = m.get("actions") or {}
    P = m.get("per_type") or {}
    return {"test MRR / chance": rb["MRR"] / max(rb.get("chance_MRR") or 1e-9, 1e-9),
            "KU types learned": sum(v["MRR_bal"] >= 1.5 * v["chance_bal"] for v in P.values()) / max(len(P), 1),
            "from structure": (st["with"] - st["without"]) if st else float("nan"),
            "action swap": _g(ac, "swap", "direction_cos"), "action flip": _g(ac, "flip", "direction_cos"),
            "train sec": m.get("train_sec")}

def board(fn, sort_col):
    rows = [{"exp": e, "arm": tag, **fn(r)} for e, res in all_results.items() for tag, runs in res.items() for r in runs]
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).apply(pd.to_numeric, errors="ignore")
    g = df.groupby(["exp", "arm"], sort=False)
    mean, std = g.mean(numeric_only=True), g.std(numeric_only=True)
    out = mean.round(3).astype(str) + std.round(3).map(lambda x: "" if x != x else f" ± {x}")
    return out.loc[mean.sort_values(sort_col, ascending=False).index]

print("Leaderboard 1 - reasoning benchmark"); display(board(reasoning_row, "reasoning index"))
print("Leaderboard 2 - retrieval and model health"); display(board(health_row, "test MRR / chance"))

### Epoch by epoch: every variation

One row per run (mixed, each type alone, ...), one column per epoch: validation balanced MRR ÷ chance at the end of that epoch (≥ 1.5 = learned). Same as the dashboard's **All runs → Epoch by epoch** matrix.

In [ ]:
for e, res in all_results.items():
    ratio, learned = {}, {}
    for tag, runs in res.items():
        for r in runs:
            key = f"{tag} (seed {r['seed']})"
            ratio[key] = {f"epoch {x['epoch']}": x["bal_MRR"] / max(x["bal_chance"], 1e-9) for x in r.get("epoch_results", [])}
            learned[key] = {f"epoch {x['epoch']}": f"{x['types_learned']}/{len(x['per_type'])}" for x in r.get("epoch_results", [])}
    if not any(ratio.values()):
        continue
    print(e, "- balanced MRR / chance:")
    display(pd.DataFrame(ratio).T.round(2).style.background_gradient(cmap="RdYlGn", vmin=0.8, vmax=2.0, axis=None))
    tk = {f"{tag} (seed {r['seed']})": __import__("sci_jepa.evaluation.tasks", fromlist=["x"]).task_summary(r.get("tasks"))
          for tag, runs in res.items() for r in runs}
    if any(tk.values()):
        print(e, "- reasoning tasks (test): retrieval x chance, consistency balanced acc., intervention toward donor:")
        display(pd.DataFrame(tk).T.round(3))
    print(e, "- KU types learned:")
    display(pd.DataFrame(learned).T)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for k, v in ratio.items():
        ax.plot([int(c.split()[1]) for c in v], list(v.values()), "o-", label=k, lw=2.5 if "mixed" in k else 1)
    ax.axhline(1, c="k", ls=":"); ax.axhline(1.5, c="g", ls="--", lw=0.8)
    ax.set_xlabel("epoch"); ax.set_ylabel("val MRR / chance"); ax.set_title(e); ax.legend(fontsize=7, ncol=2, bbox_to_anchor=(1, 1))
    plt.tight_layout(); plt.show()

### E10: does widening the neighbourhood remove the A1 degeneracy?

In [ ]:
if "E10" in all_results:
    r = all_results["E10"]
    tags = list(r)
    xs = [int(t.split("=")[1]) for t in tags]
    get = lambda k1, k2: [np.mean([x[k1][k2] for x in r[t]]) for t in tags]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3))
    ax[0].plot(xs, get("retrieval_balanced", "MRR"), "o-", label="balanced MRR")
    ax[0].plot(xs, get("retrieval_balanced", "chance_MRR"), "k:", label="chance"); ax[0].legend(); ax[0].set_xlabel("N")
    ax[1].plot(xs, get("target_space", "chi"), "o-"); ax[1].set_title("chi (type-centred)"); ax[1].set_xlabel("N")
    ax[2].plot(xs, get("partition_probe", "between_subgraph_var_share"), "o-"); ax[2].axhline(0.3, c="r", ls="--")
    ax[2].set_title("between-subgraph variance share (G1 < 0.30)"); ax[2].set_xlabel("N")
    for a in ax: a.set_xscale("log", base=2)
    plt.tight_layout(); plt.show()

## 6b. Use a trained model

Every run saves its model: the single run above to `paper_a_results/models/<run>/`, every experiment run to `paper_a_results/<experiment>/models/<arm>_seed<s>/`. Each folder holds `model.safetensors` (end of training), `best.safetensors` (best validation epoch), `config.json` and `summary.json`. `SciJEPA.from_pretrained` loads one and works on any paper given as a structured summary (the same fields as the training data):

- `embed(paper)`: one vector per knowledge unit (facet embeddings) and a paper vector
- `predict(paper, hide="result", task="forward")`: the predicted representation of a hidden part
- `rank(paper, hide, candidates, task)`: which candidate text is the most likely hidden part
- `consistency(claim, evidence)`: probability of supports vs contradicts


In [ ]:
from sci_jepa.inference import SciJEPA

api = SciJEPA.from_pretrained(MODEL_DIR, weights="best", index=gi)   # index: neighbouring papers + candidates in
                                                                     # context, as in evaluation (any folder under
                                                                     # paper_a_results/**/models/ works)
paper = {
    "research_context": "Graph neural networks struggle with long-range dependencies in molecular graphs.",
    "research_question_hypothesis": "Does adding a virtual node improve long-range message passing?",
    "methodological_details": "A virtual node connected to every atom is added to a GIN model trained on ZINC.",
    "procedures_architectures": "Five layers, hidden size 300, Adam optimiser, 200 epochs.",
    "key_results": "The virtual node reduces the mean absolute error by 15% compared with the baseline.",
    "claims": [{"description": "Virtual nodes improve long-range information flow.",
                "supporting_evidence": "Error drops by 15% on ZINC.", "contradicting_evidence": "",
                "implications": "Global context matters for molecular property prediction."}],
    "field_subfield": "Computer Science — Machine Learning",
}
print("facet embeddings:", [t for t, _ in api.embed(paper)["kus"]])
for score, text in api.rank(paper, hide="result", task="forward", candidates=[
        "The virtual node reduces the mean absolute error by 15% compared with the baseline.",
        "The virtual node increases the mean absolute error by 15% compared with the baseline.",
        "Protein structure prediction accuracy improves with more templates."]):
    print(f"{score:+.3f}  {text}")
print(api.consistency("Virtual nodes improve long-range information flow.", "Error drops by 15% on ZINC."))


## 7. Dashboard and files

In [ ]:
display(IFrame(URL, width="100%", height=800))
out = Path(os.environ["PAPER_A_OUT"])
print("dashboard      :", URL, "(served while this kernel runs)")
print("reopen later   : python -m sci_jepa.dashboard.serve", DASH)
print("KU graph cache :", os.environ["PAPER_A_INDEX_DIR"])
print("results        :", out)
for p in sorted(out.glob("*/summary.json")):
    print("  ", p.relative_to(out))